# AgentCore Browser com Assinatura Web Bot Auth 

## Visão Geral

Neste tutorial, aprenderemos como habilitar a assinatura Web Bot Auth com a ferramenta Amazon Bedrock AgentCore Browser. 

### Detalhes do Tutorial

| Informação          | Detalhes                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional                                                                   |
| Tipo de agente      | Único                                                                           |
| Framework Agêntico  | Strands                                                                          |
| Modelo LLM          | Claude 4.5 Haiku                                                                 |
| Componentes         | Automação do navegador, assinatura *Web Bot Auth* de requisições do navegador                   |
| Vertical            |                                                                                  |
| Complexidade        | Intermediário                                                                     |
| SDK utilizado       | Amazon Bedrock AgentCore Python SDK, Strands Agents, Strands Agent Tools         |
 
### Arquitetura do Tutorial

![Architecture](images/Architecture.png)

### Principais Recursos do Tutorial

* Usar a ferramenta de navegador de forma headless com agentes Strands
* Usar a ferramenta strands_tools AgentCoreBrowser
* Modelo Claude 4.5 Haiku para análise rápida e eficiente

## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS configuradas
* Amazon Bedrock AgentCore SDK
* Pacotes strands agents e strands-agents-tools
* Acesso ao modelo Claude 4.5 Haiku no Amazon Bedrock

**Implementação**: Usando o `strands_tools.browser.AgentCoreBrowser` oficial para automação do navegador a partir de um agente.

## Configuração do AgentCore Browser

Antes de usar a ferramenta AgentCore Browser, você pode opcionalmente criar uma configuração personalizada de navegador with specific settings like recording capabilities, network configuration, and execution roles. This section shows how to create a custom browser configuration using the AWS SDK.

### Criar Configuração Personalizada de Navegador

O código a seguir demonstra como criar um AgentCore Browser personalizado com gravação habilitada and a specific execution role.

#### Criar a Configuração do Navegador com Assinatura Web Bot Auth

**O que é Assinatura de Navegador?**

A assinatura de navegador (`browserSigning.enabled = True`) configura o AgentCore Browser para assinar automaticamente todas as requisições HTTP de saída. Isso é essencial para:

- **Acesso autenticado à API**: Assinar requisições para APIs protegidas
- **Integração com serviços AWS**: Assinar automaticamente requisições com credenciais AWS
- **Conformidade de segurança**: Garantir que todas as requisições do navegador sejam autenticadas
- **Integridade de requisição**: Assinar criptograficamente cabeçalhos de requisição

Quando habilitado, o navegador irá:
1. Interceptar todas as requisições HTTP/HTTPS
2. Adicionar assinaturas criptográficas aos cabeçalhos de requisição
3. Incluir tokens de autenticação automaticamente
4. Manter a integridade da sessão entre requisições


### Importar Bibliotecas


In [ ]:
import boto3
import uuid
import os
import sys
from strands import Agent
from strands_tools.browser import AgentCoreBrowser
import asyncio
import time

tutorials_path = os.path.abspath(os.path.join(os.getcwd(), '../../../'))
if tutorials_path not in sys.path:
    sys.path.insert(0, tutorials_path)

from utils import create_agentcore_role

cp_client = boto3.client('bedrock-agentcore-control', 
                         region_name='us-west-2')


accountId = boto3.client("sts").get_caller_identity()["Account"] 
region = boto3.Session().region_name

print(f"Account ID: {accountId}")
print(f"Region: {region}")

### Configuração e Criação de Role


In [ ]:
## Create the execution role.
execution_role_arn = create_agentcore_role("web-bot-auth")["Role"]["Arn"]

print(f"\n✅ Role Created Successfully : {execution_role_arn}")

## Create new browser instance with custom configurations
response = cp_client.create_browser(
    name="web_bot_auth_browser_" + str(uuid.uuid4())[:6],
    description="Browser configured to sign web bot auth",
    networkConfiguration={
        "networkMode": "PUBLIC"
    },
    executionRoleArn=execution_role_arn,
    browserSigning={
        "enabled": True
    }
)

browserId = response['browserId']
browserArn = response['browserArn']
print(f"\n✅ Browser Created Successfully!")
print(f"   Browser ID: {browserId}")
print(f"   Browser ARN: {browserArn}")
print(f"\n🔐 Browser signing is ENABLED - all requests will be automatically signed")

### Criar Agente Strands com Ferramenta AgentCoreBrowser


In [ ]:
# Create and configure the Strands agent with AgentCoreBrowser
# Initialize the official AgentCoreBrowser tool with the custom browser ID that
# was created in the previous step
agent_core_browser = AgentCoreBrowser(identifier=browserId, region="us-west-2")
agent_core_default_browser = AgentCoreBrowser(region="us-west-2")

# Import the SequentialToolExecutor to prevent concurrent browser operations
from strands.tools.executors import SequentialToolExecutor
    
# Create SIGNED agent with Claude 4.5 Haiku model and SEQUENTIAL tool execution
strands_agent = Agent(
    tools=[agent_core_browser.browser],  # Uses the custom browser with signing enabled
    tool_executor=SequentialToolExecutor(),  # This prevents concurrent browser operations
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="""You are a website analyst with browser signing capabilities.
1. Use the browser tool to visit and interact with the website EFFICIENTLY
2. Focus on extracting key information QUICKLY and within 2-3 browser interactions.
3. Review browser requests for signatures related to Web Bot Auth's Signature and Signature-Agent http headers, 
   to verify if browser signing is configured."""
)

# Create UNSIGNED agent with Claude 4.5 Haiku model and SEQUENTIAL tool execution
strands_agent_unsigned = Agent(
    tools=[agent_core_default_browser.browser],  # Uses the default browser without signing
    tool_executor=SequentialToolExecutor(),  # This prevents concurrent browser operations
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="""You are a website analyst with browser signing capabilities.
1. Use the browser tool to visit and interact with the website EFFICIENTLY
2. Focus on extracting key information QUICKLY and within 2-3 browser interactions.
3. Review browser requests for signatures related to Web Bot Auth's Signature and Signature-Agent http headers, 
   to verify if browser signing is configured."""
)

In [ ]:
# Define async function with sequential execution (no more concurrent conflicts)
async def analyze_website(agent, prompt):
    """Async wrapper for agent invocation with sequential tool execution"""
    try:
        # With SequentialToolExecutor, browser operations won't conflict
        result = await agent.invoke_async(prompt)
        return result
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

### Iniciar Análise com Assinatura de Navegador Habilitada


In [ ]:
print("\n🚀 Validate browser signing against CloudFlare's crawltest site.")
print("="*100)
  
result_signed = await analyze_website(
    strands_agent,
    "Review the output and status code at https://crawltest.com/cdn-cgi/web-bot-auth and provide 3 to 4 concise key insights, based on https://developers.cloudflare.com/bots/reference/bot-verification/web-bot-auth/"
)

if result_signed:
    print("\n\n✅ Analysis completed, with Web Bot Auth browser signing enabled")
    print("-"*100)
    print(result_signed)
    print("-"*100)

### Executar novamente o experimento sem assinatura de navegador

Para validar que nosso teste é válido, vamos executar o mesmo prompt, but with the agent that is configured without browser signing.

In [ ]:
print("\n🚀 Validate browser signing against CloudFlare's crawltest site - without Web Bot Auth browser signing.")
print("="*100)
  
result_unsigned = await analyze_website(
    strands_agent_unsigned,
    "Review the output and status code at https://crawltest.com/cdn-cgi/web-bot-auth and provide 3 to 4 concise key insights, based on https://developers.cloudflare.com/bots/reference/bot-verification/web-bot-auth/"
)

if result_unsigned:
    print("\n\n✅ Analysis completed - without Web Bot Auth browser signing")
    print("-"*100)
    print(result_unsigned)
    print("-"*100)

Vamos comparar nossos resultados:

In [ ]:
# Compare result_unsigned and result_signed using Strands Agent
comparison_prompt = f"""
Please analyze and compare these two agent outputs side by side:

**Unsigned Agent Output:**
{result_unsigned}

**Signed Agent Output:**
{result_signed}

Please provide:
* A side-by-side comparison highlighting key differences
* Validation of the Signed Agent using Signatures correctly
* Validation of the Unsigned Agent NOT using Signatures
* Summary of which aspects differ most significantly

Format your response clearly with headers and bullet points for easy reading.
"""

# Make Strands Agent call to compare the results
comparison_agent = Agent(
    system_prompt="""You are an expert data analyst evaluating different AI agent outputs.""",
    callback_handler=None
)

comparison_response = comparison_agent(comparison_prompt)

print("=== COMPARISON OF AGENT OUTPUTS ===")
print(comparison_response)


## O Que Aconteceu nos Bastidores

Quando você executa este notebook com **assinatura de navegador habilitada**, o seguinte processo ocorre:

### 1. **Criação da Configuração do Navegador**
```python
browserSigning={
    "enabled": True
}
```
Isso informa ao serviço AgentCore Browser para assinar automaticamente todas as requisições HTTP feitas pelo navegador.

### 2. **Inicialização do Agente**
O agente Strands é inicializado com:
- A browser configuration identifier with signing enabled. *Note* this is not the default browser identifier.
- Claude 4.5 Haiku model
- Browser tool integration

### 3. **Fluxo de Assinatura de Requisição**
Quando o agente navega para sites:

```text
┌─────────────────┐    ┌───────────────────┐    ┌──────────────────────┐    ┌──────────────┐
│  Agent Request  │ ──→│ AgentCore Browser │ ──→│ Sign Request Headers │ ──→│Target Website│
└─────────────────┘    └───────────────────┘    └──────────────────────┘    └──────────────┘
                                                          │
                                                          ▼
                                                Add Web Bot Auth headers:
                                                • Signature-Input header
                                                • Signature-Agent header
                                                • Signature header

```

## Benefícios de Segurança

Com assinatura de navegador habilitada:

✅ **Autenticação Automática** - Sem gerenciamento manual de cabeçalhos  
✅ **Integridade de Requisição** - Assinaturas criptográficas previnem adulteração  
✅ **Integração AWS** - Integração perfeita com credenciais IAM for agents to use the browser tool 
✅ **Gerenciamento de Sessão** - Manipulação segura de tokens de sessão with each session getting its own browser sessions 
✅ **Trilha de Auditoria** - Todas as requisições assinadas podem ser registradas  

## Solução de Problemas

### Erro: "Browser session not found"
**Solução**: The browser ID might have expired. Re-run the browser creation cell (cell-3)

### RuntimeError: "Leaving task does not match the current task"
**Solução**: This asyncio error occurs when browser operations run concurrently. The notebook uses `SequentialToolExecutor()` to prevent this issue by ensuring browser operations execute one at a time.

**Why this happens**: If your agent asks the AgentCore Browser tool performs multiple async operations (get_html, get_text, screenshot, etc.) in parallel, Strands may execute them concurrently, creating conflicting asyncio tasks.

**The fix**: Use `SequentialToolExecutor()` to ensure all tool calls execute sequentially, eliminating the async task conflicts while maintaining the same functionality.


## Leitura Adicional

Para saber mais sobre Web Bot Auth e como ele reduz CAPTCHAs para agentes de IA, confira estes recursos:

### Posts do Blog AWS
- **[Reduce CAPTCHAs for AI agents browsing the web with Web Bot Auth (Preview) in Amazon Bedrock AgentCore Browser](https://aws.amazon.com/blogs/machine-learning/reduce-captchas-for-ai-agents-browsing-the-web-with-web-bot-auth-preview-in-amazon-bedrock-agentcore-browser/)** - Comprehensive guide on Web Bot Auth implementation and benefits

### Documentação
- **[Cloudflare Web Bot Auth Documentation](https://developers.cloudflare.com/bots/reference/bot-verification/web-bot-auth/)** - Technical specification and implementation details
- **[Amazon Bedrock AgentCore Browser Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-onboarding.html)** - Complete guide to AgentCore Browser features

### Tópicos Relacionados
- **[HTTP Message Signatures (RFC 9421)](https://datatracker.ietf.org/doc/rfc9421/)** - The underlying cryptographic standard used by Web Bot Auth
- **[Web Bot Auth Architecture (IETF Draft)](https://datatracker.ietf.org/doc/html/draft-meunier-web-bot-auth-architecture)** - The IETF draft specification for Web Bot Auth architecture